# AML Benchmark — Large Dataset Run

**Vorbereitung lokal (einmalig, bevor du dieses Notebook startest):**
1. Folgende Files auf Google Drive hochladen in `MyDrive/aml_data/`:
   - `LI-Large_Trans.csv`
   - `LI-Large_accounts.csv`
   - `LI-Large_Patterns.txt`
2. Code ist auf GitHub: https://github.com/fdrmic/classimbalance

## Schritt 1 — Google Drive mounten

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive gemountet!')

## Schritt 2 — RAM & GPU prüfen

In [ ]:
import psutil, os
ram = psutil.virtual_memory()
print(f'Gesamt-RAM : {ram.total / 1e9:.1f} GB')
print(f'Freier RAM : {ram.available / 1e9:.1f} GB')
os.system('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader')

## Schritt 3 — Projektcode von GitHub klonen

In [ ]:
import os
from pathlib import Path

!git clone https://github.com/fdrmic/classimbalance.git /content/classimbalance

PROJECT_DIR = Path('/content/classimbalance')
os.chdir(PROJECT_DIR)
print('Arbeitsverzeichnis:', os.getcwd())
print('Inhalt:', sorted(os.listdir('.')))

## Schritt 4 — Dependencies installieren

In [ ]:
!pip install -e . -q
!pip install -r requirements.txt -q
print('Installation abgeschlossen!')

## Schritt 5 — Large Files auf Drive prüfen

In [ ]:
from pathlib import Path

# ANPASSEN falls dein Drive-Ordner anders heisst
DRIVE_DIR = Path('/content/drive/MyDrive/aml_data')

for fname in ['LI-Large_Trans.csv', 'LI-Large_accounts.csv', 'LI-Large_Patterns.txt']:
    p = DRIVE_DIR / fname
    if p.exists():
        print(f'  OK    {fname}  ({p.stat().st_size / 1e9:.2f} GB)')
    else:
        print(f'  FEHLT {fname}  <- Upload prüfen!')

## Schritt 6 — paths_large.yaml erstellen

In [ ]:
import yaml, os
from pathlib import Path

PROJECT_DIR = Path(os.getcwd())
DRIVE_DIR   = Path('/content/drive/MyDrive/aml_data')  # anpassen falls nötig

with open(PROJECT_DIR / 'configs' / 'paths.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['transactions_filename'] = 'LI-Large_Trans.csv'
cfg['accounts_filename']     = 'LI-Large_accounts.csv'
cfg['patterns_filename']     = 'LI-Large_Patterns.txt'
cfg['raw_dir']               = str(DRIVE_DIR)
cfg['processed_dir']         = str(PROJECT_DIR / 'data' / 'processed')
cfg['splits_dir']            = str(PROJECT_DIR / 'data' / 'splits')
cfg['outputs_dir']           = str(PROJECT_DIR / 'outputs' / 'runs')
cfg['leaderboard_dir']       = str(PROJECT_DIR / 'outputs' / 'leaderboard')

out_path = PROJECT_DIR / 'configs' / 'paths_large.yaml'
with open(out_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('paths_large.yaml erstellt:')
print(yaml.dump(cfg, default_flow_style=False))

## Schritt 7 — Daten einlesen & labeln
**Überspringen falls `transactions_labeled.parquet` bereits auf Drive vorhanden und in Schritt 6 eingebunden.**

In [ ]:
!python -m aml_benchmark.data.make_dataset --paths configs/paths_large.yaml

## Schritt 8 — Zeitliche Splits erstellen
**Überspringen falls `split_manifest.json` bereits vorhanden.**

In [ ]:
!python -m aml_benchmark.data.splitter --paths configs/paths_large.yaml

## Schritt 9 — 30-Run-Benchmark (Part A)
⚠️ Dauert mehrere Stunden!

In [ ]:
!python -m aml_benchmark.experiments.grid_runner \
    --paths configs/paths_large.yaml

## Schritt 10 — Threshold-Optimierung

In [ ]:
!python -m aml_benchmark.experiments.re_evaluate --paths configs/paths_large.yaml

## Schritt 11 — Leaderboard aggregieren

In [ ]:
!python -m aml_benchmark.experiments.aggregate --paths configs/paths_large.yaml

## Schritt 12 — Ergebnisse auf Drive sichern
⚠️ Immer ausführen bevor die Session endet!

In [ ]:
import shutil, datetime, os
from pathlib import Path

PROJECT_DIR = Path(os.getcwd())
ts = datetime.datetime.now().strftime('%Y%m%d_%H%M')
backup_dir = Path(f'/content/drive/MyDrive/aml_results/large_run_{ts}')
backup_dir.mkdir(parents=True, exist_ok=True)

to_backup = {
    'outputs/leaderboard' : 'leaderboard',
    'outputs/runs'        : 'runs',
    'data/processed'      : 'processed',
    'data/splits'         : 'splits',
}

for src_rel, dst_name in to_backup.items():
    src = PROJECT_DIR / src_rel
    if src.exists():
        shutil.copytree(src, backup_dir / dst_name, dirs_exist_ok=True)
        print(f'Gesichert: {src_rel}/')
    else:
        print(f'Nicht gefunden (OK falls noch nicht erstellt): {src_rel}/')

print(f'\nBackup abgeschlossen: {backup_dir}')

---
## Troubleshooting

**`--paths` Argument wird nicht erkannt:**  
Falls ein Modul `--paths` nicht kennt, melde dich — dann passen wir die Entry-Points an.

**`MemoryError` bei make_dataset:**  
Melde dich — dann wird `load_transactions()` auf chunked loading umgeschrieben.

**Code wurde lokal geändert aber nicht auf GitHub gepusht:**  
Lokal `git add . && git commit -m 'update' && git push` ausführen, dann in Colab `!git pull` in Schritt 3 ergänzen.

**Session läuft ab:**  
Schritt 12 sichert `processed/` und `splits/` auf Drive. Beim nächsten Start Schritte 7+8 überspringen.